# Plate 01 · Dataset Preparation

Builds one unified detection dataset (YOLO format) and one recognition dataset (crop + text).

| Source | Use | Licence note |
|---|---|---|
| **EALPR** (Egyptian ALPR, ~2.4k images, letters + digits boxes) | detector + recogniser | research use – check the dataset page |
| Kaggle *Egyptian car plates* sets | detector + recogniser | per dataset |
| **Open Images v7** class *Vehicle registration plate* | detector pre-training | CC-BY 4.0 |
| Synthetic plates (notebook 02) | recogniser, rare plate types, night/rain | ours |

Split by **plate text** (`stable_split`) so the same car never appears in train and test.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import json, shutil
import pandas as pd
from Common_Utils.Data_Utils import list_images, stable_split

RAW = DATA / "Plates" / "Raw"
DET = DATA / "Plates" / "Detection"      # YOLO format
REC = DATA / "Plates" / "Recognition"    # crops + labels.csv
for d in (RAW, DET, REC):
    d.mkdir(parents=True, exist_ok=True)

### 1. Download (fill in the dataset slugs you have accepted)

In [ ]:
KAGGLE_SLUGS = [
    # "owner/egyptian-license-plates",   # <- put the exact slug(s) here
]
import subprocess
for slug in KAGGLE_SLUGS:
    target = RAW / slug.split("/")[-1]
    if not target.exists():
        subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(target), "--unzip"], check=True)
print(sorted(p.name for p in RAW.iterdir()))

### 2. Convert annotations to YOLO
Each source has its own format; write one small adapter per source that yields
`(image_path, [(class_id, x_center, y_center, w, h) ...], plate_text)` in normalised coordinates.
Classes: `0 private, 1 taxi, 2 transport, 3 commercial, 4 other` (colour band of the plate).

In [ ]:
PLATE_CLASSES = ["private", "taxi", "transport", "commercial", "other"]

def adapter_example(source_dir):
    """Template adapter - replace with the real format of each dataset (VOC xml, json, csv...)."""
    for label_file in sorted(Path(source_dir).rglob("*.txt")):
        image = label_file.with_suffix(".jpg")
        if not image.exists():
            continue
        boxes = [tuple(map(float, line.split()[:5])) for line in label_file.read_text().splitlines() if line.strip()]
        yield image, [(int(b[0]), *b[1:]) for b in boxes], label_file.stem

def write_yolo(records):
    rows = []
    for image, boxes, text in records:
        split = stable_split(text)
        (DET / "images" / split).mkdir(parents=True, exist_ok=True)
        (DET / "labels" / split).mkdir(parents=True, exist_ok=True)
        dst = DET / "images" / split / image.name
        shutil.copy2(image, dst)
        (DET / "labels" / split / f"{image.stem}.txt").write_text(
            "\n".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h in boxes))
        rows.append({"image": dst.name, "split": split, "boxes": len(boxes), "text": text})
    return pd.DataFrame(rows)

frames = [write_yolo(adapter_example(src)) for src in RAW.iterdir() if src.is_dir()]
index = pd.concat(frames) if frames else pd.DataFrame(columns=["image", "split", "boxes", "text"])
index.to_csv(DET / "index.csv", index=False)
index.groupby("split").size() if len(index) else "no data yet"

In [ ]:
(DET / "plates.yaml").write_text(f"""path: {DET.as_posix()}
train: images/train
val: images/val
test: images/test
names: {json.dumps(PLATE_CLASSES)}
""")
print((DET / "plates.yaml").read_text())

### 3. Recognition crops
Crop every labelled plate (with 8 % padding), resize to **height 48**, save with its text.
Normalise the label with `Plate_Normalizer` *only for matching* – the recogniser learns the Arabic display form.

In [ ]:
import cv2
from Common_Utils.Plate_Normalizer import to_western_digits

def crop_plates(det_root=DET, out=REC, pad=0.08):
    rows = []
    for label in (det_root / "labels").rglob("*.txt"):
        split = label.parent.name
        image = cv2.imread(str(det_root / "images" / split / f"{label.stem}.jpg"))
        if image is None:
            continue
        h, w = image.shape[:2]
        for i, line in enumerate(label.read_text().splitlines()):
            c, xc, yc, bw, bh = map(float, line.split()[:5])
            x1, y1 = int((xc - bw / 2 - pad * bw) * w), int((yc - bh / 2 - pad * bh) * h)
            x2, y2 = int((xc + bw / 2 + pad * bw) * w), int((yc + bh / 2 + pad * bh) * h)
            crop = image[max(0, y1):y2, max(0, x1):x2]
            if crop.size == 0:
                continue
            crop = cv2.resize(crop, (int(48 * crop.shape[1] / crop.shape[0]), 48))
            name = f"{label.stem}_{i}.png"
            (out / split).mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(out / split / name), crop)
            rows.append({"file": f"{split}/{name}", "split": split, "text": label.stem})  # replace with real text
    df = pd.DataFrame(rows)
    df.to_csv(out / "labels.csv", index=False)
    return df

crops = crop_plates()
len(crops)